## Terra Bank Loan Risk Analysis 
#### Phase 1: Data Cleaning and Preprocessing

#### About Dataset:

***Source*** Kaggle - [Bank Loan Data] [https://www.kaggle.com/datasets/udaymalviya/bank-loan-data]

*** Data Description *** :

- 45,000 rows each representing a unique personal bank loan
- 14 columns covering client demographics, financial profile, loan characteristics, and credit history. 
- each record corresponnds to a closed loan with an outcome of either fully prepaid or defaulted 

#### Project Goal:

- To Identify and better understand which applicant and loan features are most strongly associated with a loan being repaid versus defaulted
- Engineer a predictive model that cam help with reducing default risk, adjusting approval thresholds, etc.



### Objectives
- Import necessary libraries and packages
- Load and examine the associated dataset
- Initial dataset cleaning and preparation for Preliminary Analysis 


# Bank Loan Data Cleaning and Preprocessing

This notebook loads the raw bank loan dataset, standardizes the schema, checks for duplicates, profiles numeric variables, flags outliers with rule-based and distribution-based methods, documents cleaning decisions, and saves both an audit-friendly full dataset and a modeling-ready cleaned dataset.


## Project configuration


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

TARGET_COL = "loan_repaid"
NUM_COLS = [
    "age",
    "income",
    "loan_amount",
    "loan_interest_rate",
    "loan_percent_to_income",
    "credit_history_length",
    "credit_score",
]

FULL_OUTPUT = INTERIM_DIR / "bank_loan_data_v2_full.csv"
MODEL_OUTPUT = INTERIM_DIR / "bank_loan_data_v2_model.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim dir: {INTERIM_DIR}")


Project root: /Users/macbook/projects/data-science/bank-loan-risk
Interim dir: /Users/macbook/projects/data-science/bank-loan-risk/data/interim


## Reusable cleaning utilities


In [2]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    rename_map = {"person_gender": "gender"}
    return df.rename(columns=rename_map)


def add_rule_outlier_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    max_age = 90
    min_age = 18
    max_income = 500_000
    max_experience_buffer = 14

    df["age_outlier_rule"] = (df["age"] < min_age) | (df["age"] > max_age)
    df["income_outlier_rule"] = df["income"] > max_income
    df["experience_outlier_rule"] = df["employment_experience_length"] > (df["age"] - max_experience_buffer)
    df["credit_history_outlier_rule"] = df["credit_history_length"] > (df["age"] - 16)
    return df


def add_iqr_outlier_flags(df: pd.DataFrame, num_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in num_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df[f"{col}_outlier_iqr"] = (df[col] < lower) | (df[col] > upper)
    return df


def add_zscore_outlier_flags(df: pd.DataFrame, z_cols: list[str], threshold: float = 3.0) -> pd.DataFrame:
    df = df.copy()
    for col in z_cols:
        zscores = stats.zscore(df[col].astype(float), nan_policy="omit")
        df[f"{col}_outlier_z"] = np.abs(zscores) > threshold
    return df


def summarize_iqr_flags(df: pd.DataFrame, num_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in num_cols:
        flag_col = f"{col}_outlier_iqr"
        rows.append({
            "feature": col,
            "iqr_outlier_proportion": df[flag_col].mean(),
            "iqr_outlier_count": int(df[flag_col].sum())
        })
    return pd.DataFrame(rows).sort_values("iqr_outlier_proportion", ascending=False).reset_index(drop=True)


def summarize_all_outliers(df: pd.DataFrame, num_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in num_cols:
        row = {"feature": col}
        rule_col = f"{col}_outlier_rule"
        iqr_col = f"{col}_outlier_iqr"
        z_col = f"{col}_outlier_z"
        row["rule_outlier_pct"] = round(df[rule_col].mean() * 100, 2) if rule_col in df.columns else np.nan
        row["iqr_outlier_pct"] = round(df[iqr_col].mean() * 100, 2) if iqr_col in df.columns else np.nan
        row["z_outlier_pct"] = round(df[z_col].mean() * 100, 2) if z_col in df.columns else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


In [3]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib
import xgboost as xgb
import os
from pathlib import Path

# current working directory is notebooks/cleaning_eda
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # go up two levels: cleaning_eda -> notebooks -> project root

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "bank_loan_data.csv"

df = pd.read_csv(DATA_FILE)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [5]:
# clean column names: strip whitespace, convert to lowercase, replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [6]:
# check for duplicates

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    display(df[df.duplicated(keep=False)].sort_index())

Duplicate rows: 0


In [7]:
# check for missing values

df.isnull().sum()

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

In [8]:
new_column_names = {
    "person_age": "age",
    "Person_gender": "gender",
    "person_income": "income",
    "person_education": "education",
    "person_emp_exp": "employment_experience_length",
    "person_home_ownership": "home_ownership",
    "loan_amnt": "loan_amount",
    "loan_intent": "loan_purpose",
    "loan_int_rate": "loan_interest_rate",
    "loan_percent_income": "loan_percent_to_income",
    "cb_person_cred_hist_length": "credit_history_length",
    "credit_score": "credit_score",
    "previous_loan_defaults_on_file": "previous_loan_default",
    "loan_status": "loan_repaid" 
}

df = df.rename(columns=new_column_names)

In [9]:
df.head()

,age,person_gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,credit_history_length,credit_score,previous_loan_default,loan_repaid
0,22.0000,female,Master,"71,948.0000",0,RENT,"35,000.0000",PERSONAL,16.0200,0.4900,3.0000,561,No,1
1,21.0000,female,High School,"12,282.0000",0,OWN,"1,000.0000",EDUCATION,11.1400,0.0800,2.0000,504,Yes,0
2,25.0000,female,High School,"12,438.0000",3,MORTGAGE,"5,500.0000",MEDICAL,12.8700,0.4400,3.0000,635,No,1
3,23.0000,female,Bachelor,"79,753.0000",0,RENT,"35,000.0000",MEDICAL,15.2300,0.4400,2.0000,675,No,1
4,24.0000,male,Master,"66,135.0000",1,RENT,"35,000.0000",MEDICAL,14.2700,0.5300,4.0000,586,No,1


#### Cleaning Findings

- all columns have been standardized 
- no duplicates detected
- no missing values detected 
- cleaned column names for clarity 
- target feature is "loan_repaid" with 1 being yes and 0 being the loan defaulted

In [10]:
# get summary statistics for numeric columns

df.describe()

,age,income,employment_experience_length,loan_amount,loan_interest_rate,loan_percent_to_income,credit_history_length,credit_score,loan_repaid
count,"45,000.0000","45,000.0000","45,000.0000","45,000.0000","45,000.0000","45,000.0000","45,000.0000","45,000.0000","45,000.0000"
mean,27.7642,"80,319.0532",5.4103,"9,583.1576",11.0066,0.1397,5.8675,632.6088,0.2222
std,6.0451,"80,422.4986",6.0635,"6,314.8867",2.9788,0.0872,3.8797,50.4359,0.4157
min,20.0000,"8,000.0000",0.0000,500.0000,5.4200,0.0000,2.0000,390.0000,0.0000
25%,24.0000,"47,204.0000",1.0000,"5,000.0000",8.5900,0.0700,3.0000,601.0000,0.0000
50%,26.0000,"67,048.0000",4.0000,"8,000.0000",11.0100,0.1200,4.0000,640.0000,0.0000
75%,30.0000,"95,789.2500",8.0000,"12,237.2500",12.9900,0.1900,8.0000,670.0000,0.0000
max,144.0000,"7,200,766.0000",125.0000,"35,000.0000",20.0000,0.6600,30.0000,850.0000,1.0000


In [11]:
# calculate mean for numeric columns, round to 2 decimal places and create a table 

mean_table = df.mean(numeric_only=True).to_frame(name="mean").round(2)
mean_table

,mean
age,27.7600
income,"80,319.0500"
employment_experience_length,5.4100
loan_amount,"9,583.1600"
loan_interest_rate,11.0100
loan_percent_to_income,0.1400
credit_history_length,5.8700
credit_score,632.6100
loan_repaid,0.2200


In [12]:
df.head(10)

,age,person_gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,credit_history_length,credit_score,previous_loan_default,loan_repaid
0,22.0000,female,Master,"71,948.0000",0,RENT,"35,000.0000",PERSONAL,16.0200,0.4900,3.0000,561,No,1
1,21.0000,female,High School,"12,282.0000",0,OWN,"1,000.0000",EDUCATION,11.1400,0.0800,2.0000,504,Yes,0
2,25.0000,female,High School,"12,438.0000",3,MORTGAGE,"5,500.0000",MEDICAL,12.8700,0.4400,3.0000,635,No,1
3,23.0000,female,Bachelor,"79,753.0000",0,RENT,"35,000.0000",MEDICAL,15.2300,0.4400,2.0000,675,No,1
4,24.0000,male,Master,"66,135.0000",1,RENT,"35,000.0000",MEDICAL,14.2700,0.5300,4.0000,586,No,1
5,21.0000,female,High School,"12,951.0000",0,OWN,"2,500.0000",VENTURE,7.1400,0.1900,2.0000,532,No,1
6,26.0000,female,Bachelor,"93,471.0000",1,RENT,"35,000.0000",EDUCATION,12.4200,0.3700,3.0000,701,No,1
7,24.0000,female,High School,"95,550.0000",5,RENT,"35,000.0000",MEDICAL,11.1100,0.3700,4.0000,585,No,1
8,24.0000,female,Associate,"100,684.0000",3,RENT,"35,000.0000",PERSONAL,8.9000,0.3500,2.0000,544,No,1
9,21.0000,female,High School,"12,739.0000",0,OWN,"1,600.0000",VENTURE,14.7400,0.1300,3.0000,640,No,1


## Duplicate detection note

A composite key is used here because the dataset does not include a dedicated loan ID. The key combines borrower profile and loan terms to approximate a duplicate record check.


In [13]:
dup_rate = df["is_dup_key"].mean() if "is_dup_key" in df.columns else 0
print(f"Duplicate-key rate: {dup_rate:.4%}")


Duplicate-key rate: 0.0000%


## Standardize schema


In [14]:
df = standardize_column_names(df)
print(df.columns.tolist())


['age', 'gender', 'education', 'income', 'employment_experience_length', 'home_ownership', 'loan_amount', 'loan_purpose', 'loan_interest_rate', 'loan_percent_to_income', 'credit_history_length', 'credit_score', 'previous_loan_default', 'loan_repaid']


## Numeric profiling


In [15]:
numeric_profile = df[NUM_COLS + [TARGET_COL]].describe().T
display(numeric_profile)


,count,mean,std,min,25%,50%,75%,max
age,"45,000.0000",27.7642,6.0451,20.0000,24.0000,26.0000,30.0000,144.0000
income,"45,000.0000","80,319.0532","80,422.4986","8,000.0000","47,204.0000","67,048.0000","95,789.2500","7,200,766.0000"
loan_amount,"45,000.0000","9,583.1576","6,314.8867",500.0000,"5,000.0000","8,000.0000","12,237.2500","35,000.0000"
loan_interest_rate,"45,000.0000",11.0066,2.9788,5.4200,8.5900,11.0100,12.9900,20.0000
loan_percent_to_income,"45,000.0000",0.1397,0.0872,0.0000,0.0700,0.1200,0.1900,0.6600
credit_history_length,"45,000.0000",5.8675,3.8797,2.0000,3.0000,4.0000,8.0000,30.0000
credit_score,"45,000.0000",632.6088,50.4359,390.0000,601.0000,640.0000,670.0000,850.0000
loan_repaid,"45,000.0000",0.2222,0.4157,0.0000,0.0000,0.0000,0.0000,1.0000


## Rule-based outlier flags


In [16]:
df = add_rule_outlier_flags(df)

rule_flags = [
    "age_outlier_rule",
    "income_outlier_rule",
    "experience_outlier_rule",
    "credit_history_outlier_rule",
]

rule_flag_summary = (
    df[rule_flags]
    .mean()
    .sort_values(ascending=False)
    .to_frame(name="flagged_proportion")
)

rule_flag_counts = df[rule_flags].sum().to_frame(name="flagged_count")

display(rule_flag_summary)
display(rule_flag_counts)


,flagged_proportion
income_outlier_rule,0.0023
age_outlier_rule,0.0002
experience_outlier_rule,0.0000
credit_history_outlier_rule,0.0000


,flagged_count
age_outlier_rule,8
income_outlier_rule,104
experience_outlier_rule,0
credit_history_outlier_rule,0


## IQR and Z-score outlier flags


In [24]:
df = add_iqr_outlier_flags(df, NUM_COLS)
df = add_zscore_outlier_flags(df, ["credit_score", "loan_interest_rate", "age"])




## Outlier Findings Summary Table

In [25]:
iqr_summary = summarize_iqr_flags(df, NUM_COLS)
outlier_summary = summarize_all_outliers(df, NUM_COLS)

display(iqr_summary)
display(outlier_summary)

,feature,iqr_outlier_proportion,iqr_outlier_count
0,loan_amount,0.0522,2348
1,income,0.0493,2218
2,age,0.0486,2188
3,credit_history_length,0.0304,1366
4,loan_percent_to_income,0.0165,744
5,credit_score,0.0104,467
6,loan_interest_rate,0.0028,124


,feature,rule_outlier_pct,iqr_outlier_pct,z_outlier_pct
0,age,0.0200,4.8600,1.6900
1,income,0.2300,4.9300,NaN
2,loan_amount,NaN,5.2200,NaN
3,loan_interest_rate,NaN,0.2800,0.1900
4,loan_percent_to_income,NaN,1.6500,NaN
5,credit_history_length,NaN,3.0400,NaN
6,credit_score,NaN,1.0400,0.5200


## Cleaning decisions


In [18]:
decision_table = pd.DataFrame({
    "feature": [
        "age",
        "income",
        "loan_amount",
        "loan_interest_rate",
        "loan_percent_to_income",
        "credit_history_length",
        "credit_score",
    ],
    "recommended_action": [
        "Drop impossible rule-based records; keep plausible tails.",
        "Keep and consider capping or log-transforming for modeling.",
        "Keep and consider capping or log-transforming for modeling.",
        "Keep as-is; very low extreme rate.",
        "Keep; optionally cap high-ratio tail and create a high-DTI flag.",
        "Keep unless impossible relative to borrower age.",
        "Keep as-is; a few extremes are still informative in credit risk.",
    ]
})

display(decision_table)


,feature,recommended_action
0,age,Drop impossible rule-based records; keep plaus...
1,income,Keep and consider capping or log-transforming ...
2,loan_amount,Keep and consider capping or log-transforming ...
3,loan_interest_rate,Keep as-is; very low extreme rate.
4,loan_percent_to_income,Keep; optionally cap high-ratio tail and creat...
5,credit_history_length,Keep unless impossible relative to borrower age.
6,credit_score,Keep as-is; a few extremes are still informati...


## Build modeling-clean dataset


In [19]:
impossible_mask = (
    df["age_outlier_rule"]
    | df["experience_outlier_rule"]
    | df["credit_history_outlier_rule"]
)

impossible_rows = df[impossible_mask].copy()
df_clean = df[~impossible_mask].copy()

for col in ["income", "loan_amount", "loan_percent_to_income"]:
    upper_cap = df_clean[col].quantile(0.99)
    df_clean[f"{col}_capped"] = df_clean[col].clip(upper=upper_cap)

print(f"Impossible rows flagged for removal: {impossible_rows.shape[0]}")
print(f"df shape: {df.shape}")
print(f"df_clean shape: {df_clean.shape}")

display(df_clean.head())

Impossible rows flagged for removal: 8
df shape: (45000, 28)
df_clean shape: (44992, 31)


,age,gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,credit_history_length,credit_score,previous_loan_default,loan_repaid,age_outlier_rule,income_outlier_rule,experience_outlier_rule,credit_history_outlier_rule,age_outlier_iqr,income_outlier_iqr,loan_amount_outlier_iqr,loan_interest_rate_outlier_iqr,loan_percent_to_income_outlier_iqr,credit_history_length_outlier_iqr,credit_score_outlier_iqr,credit_score_outlier_z,loan_interest_rate_outlier_z,age_outlier_z,income_capped,loan_amount_capped,loan_percent_to_income_capped
0,22.0000,female,Master,"71,948.0000",0,RENT,"35,000.0000",PERSONAL,16.0200,0.4900,3.0000,561,No,1,False,False,False,False,False,False,True,False,True,False,False,False,False,False,"71,948.0000","28,393.0600",0.4000
1,21.0000,female,High School,"12,282.0000",0,OWN,"1,000.0000",EDUCATION,11.1400,0.0800,2.0000,504,Yes,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,"12,282.0000","1,000.0000",0.0800
2,25.0000,female,High School,"12,438.0000",3,MORTGAGE,"5,500.0000",MEDICAL,12.8700,0.4400,3.0000,635,No,1,False,False,False,False,False,False,False,False,True,False,False,False,False,False,"12,438.0000","5,500.0000",0.4000
3,23.0000,female,Bachelor,"79,753.0000",0,RENT,"35,000.0000",MEDICAL,15.2300,0.4400,2.0000,675,No,1,False,False,False,False,False,False,True,False,True,False,False,False,False,False,"79,753.0000","28,393.0600",0.4000
4,24.0000,male,Master,"66,135.0000",1,RENT,"35,000.0000",MEDICAL,14.2700,0.5300,4.0000,586,No,1,False,False,False,False,False,False,True,False,True,False,False,False,False,False,"66,135.0000","28,393.0600",0.4000


In [23]:
print("Impossible rows removed:", impossible_rows.shape[0])
print("Original rows:", df.shape[0])
print("Clean rows:", df_clean.shape[0])

# Check default rate stability
print("Default rate original:", 1 - df["loan_repaid"].mean())
print("Default rate clean:", 1 - df_clean["loan_repaid"].mean())


Impossible rows removed: 8
Original rows: 45000
Clean rows: 44992
Default rate original: 0.7777777777777778
Default rate clean: 0.7777382645803699


### Important Actions for Flags
- We decided to drop a total of 8 rows with impossible age/experience/credit history combinations 
- We also capped Income, Loan Amount and Loan Percent to Income at the 99th percentile 

## Save outputs


In [20]:
df.to_csv(FULL_OUTPUT, index=False)
df_clean.to_csv(MODEL_OUTPUT, index=False)

print(f"Saved full dataset to: {FULL_OUTPUT}")
print(f"Saved modeling dataset to: {MODEL_OUTPUT}")


Saved full dataset to: /Users/macbook/projects/data-science/bank-loan-risk/data/interim/bank_loan_data_v2_full.csv
Saved modeling dataset to: /Users/macbook/projects/data-science/bank-loan-risk/data/interim/bank_loan_data_v2_model.csv
